# LeetCode 1054 — Distant Barcodes bằng Python

Link đề: https://leetcode.com/problems/distant-barcodes/ · Độ khó: **Medium** · Chủ đề: Greedy (tham lam), Array (mảng), Hash Table (bảng băm), Counting (đếm), Sorting (sắp xếp), Heap (hàng đợi ưu tiên)

Skeleton LeetCode:

```python
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
        ...
```

Ghi chú: sắp xếp lại một dãy mã vạch sao cho **không có hai mã vạch kề nhau bằng nhau**. Đề bảo luôn có đáp án và chấp nhận **bất kỳ** đáp án đúng. Ý tưởng chính: xếp các giá trị **xuất hiện nhiều nhất trước**, rải lần lượt lên các vị trí chẵn `0, 2, 4, ...`, hết vị trí chẵn thì sang các vị trí lẻ `1, 3, 5, ...`; mỗi giá trị xếp liền nhau trong thứ tự điền nên không thể đụng nhau. Vì có **nhiều đáp án đúng**, ta kiểm tra bằng **hàm kiểm tra tính hợp lệ** (cùng bội số phần tử và không có hai phần tử kề nhau bằng nhau), không so sánh với một mảng cố định.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề và **điều kiện tồn tại**: đáp án tồn tại khi và chỉ khi giá trị xuất hiện nhiều nhất có tần suất `<= ceil(n / 2)`.
2. Nắm các công cụ Python cần thiết: `collections.Counter`, `sorted` với `key` nhiều tiêu chí, `heapq` (hàng đợi ưu tiên, dùng số âm để mô phỏng max-heap), gán lát cắt mở rộng `res[0::2] = ...`.
3. Hiểu kỹ thuật **rải vào vị trí chẵn rồi lẻ**: giá trị nào cũng chiếm một đoạn liên tiếp trong thứ tự điền nên hai bản sao cạnh nhau cách nhau ít nhất `ceil(n / 2) - 1` ô trong thứ tự điền.
4. Đi qua 4 cách: quay lui (brute force, dùng cho dữ liệu rất nhỏ), tham lam bằng heap, đếm + sắp xếp theo tần suất rồi điền chẵn-lẻ (cách nộp chính), và chỉ cần đặt giá trị phổ biến nhất trước.
5. **Kiểm tra bằng hàm hợp lệ** vì bài có nhiều đáp án đúng.
6. Tránh các lỗi kinh điển: tham lam theo giá trị nhỏ nhất, không sắp xếp theo tần suất, sắp theo tần suất nhưng không gom giá trị trùng, điền vị trí lẻ trước, quên đẩy lại phần tử vừa dùng ở heap, và dùng `n // 2` thay vì `(n + 1) // 2` khi chia đôi.

## 2. Đề bài & ví dụ

Trong kho có một hàng mã vạch, mã vạch thứ `i` là `barcodes[i]`.

Hãy sắp xếp lại các mã vạch sao cho **không có hai mã vạch kề nhau bằng nhau**. Bạn có thể trả về **bất kỳ** đáp án nào; đề đảm bảo luôn có đáp án.

**Ví dụ 1:** `barcodes = [1,1,1,2,2,2]` -> `[2,1,2,1,2,1]` (hoặc `[1,2,1,2,1,2]`).

**Ví dụ 2:** `barcodes = [1,1,1,1,2,2,3,3]` -> `[1,3,1,3,1,2,1,2]` (nhiều đáp án khác cũng đúng).

**Ràng buộc:**

- `1 <= barcodes.length <= 10000`
- `1 <= barcodes[i] <= 10000`

In [1]:
from typing import List
from collections import Counter
from itertools import permutations, product
import heapq, random, time

def is_valid(barcodes: List[int], res: List[int]) -> bool:
    """Hợp lệ nếu res là hoán vị của barcodes và không có hai phần tử kề nhau bằng nhau."""
    if not isinstance(res, list) or len(res) != len(barcodes):
        return False
    if Counter(res) != Counter(barcodes):
        return False
    return all(res[k] != res[k + 1] for k in range(len(res) - 1))

examples = [
    [1, 1, 1, 2, 2, 2],
    [1, 1, 1, 1, 2, 2, 3, 3],
]
doc_outputs = [[2, 1, 2, 1, 2, 1], [1, 3, 1, 3, 1, 2, 1, 2]]
for barcodes, out in zip(examples, doc_outputs):
    print(f"barcodes={barcodes}: đáp án mẫu của đề {out} hợp lệ? {is_valid(barcodes, out)}")
    assert is_valid(barcodes, out)

barcodes=[1, 1, 1, 2, 2, 2]: đáp án mẫu của đề [2, 1, 2, 1, 2, 1] hợp lệ? True
barcodes=[1, 1, 1, 1, 2, 2, 3, 3]: đáp án mẫu của đề [1, 3, 1, 3, 1, 2, 1, 2] hợp lệ? True


## 3. Kiến thức nền

### 3.1. `Counter` và sắp xếp nhiều tiêu chí

`Counter(barcodes)` đếm số lần xuất hiện. `sorted(cnt, key=lambda x: (-cnt[x], x))` sắp các giá trị **theo tần suất giảm dần, rồi theo giá trị tăng dần** (dùng số âm để đảo chiều).

In [2]:
cnt = Counter([3, 1, 3, 2, 3, 1])
print(cnt)
print(sorted(cnt, key=lambda x: (-cnt[x], x)))
print(sorted(cnt, key=lambda x: cnt[x]))                 # tăng dần theo tần suất
print(cnt.most_common(1))                                # (giá trị, tần suất) phổ biến nhất

Counter({3: 3, 1: 2, 2: 1})
[3, 1, 2]
[2, 1, 3]
[(3, 3)]


### 3.2. Gán theo bước nhảy

Ta có thể ghi vào các vị trí chẵn rồi lẻ bằng chỉ số: `i = 0; ...; i += 2; if i >= n: i = 1`. Hoặc dùng gán lát cắt mở rộng `res[0::2] = ...` (độ dài hai vế **phải bằng nhau**, nếu không Python báo `ValueError`).

In [3]:
n = 7
res = [0] * n
print("các vị trí chẵn:", list(range(0, n, 2)), "| các vị trí lẻ:", list(range(1, n, 2)))
res[0::2] = ["a", "b", "c", "d"]
res[1::2] = ["x", "y", "z"]
print(res)

các vị trí chẵn: [0, 2, 4, 6] | các vị trí lẻ: [1, 3, 5]
['a', 'x', 'b', 'y', 'c', 'z', 'd']


### 3.3. `heapq` (max-heap bằng số âm)

`heapq` chỉ có min-heap. Để lấy phần tử có tần suất lớn nhất ta lưu `(-tần suất, giá trị)`; `heappop` trả về phần tử nhỏ nhất, tức tần suất lớn nhất.

In [4]:
h = [(-c, v) for v, c in Counter([3, 1, 3, 2, 3, 1]).items()]
heapq.heapify(h)
print(heapq.heappop(h), heapq.heappop(h), heapq.heappop(h))

(-3, 3) (-2, 1) (-1, 2)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `[1,1,1,2,2,2]`, `n = 6`. Có `3` số `1` và `3` số `2`; xen kẽ `1,2,1,2,1,2` là đủ.

Ví dụ 2: `[1,1,1,1,2,2,3,3]`, `n = 8`. Số `1` xuất hiện `4` lần, đúng một nửa. Ta đặt các số `1` vào vị trí chẵn `0, 2, 4, 6`; các số còn lại vào vị trí lẻ `1, 3, 5, 7`: `[1,?,1,?,1,?,1,?]` rồi điền `2,2,3,3` ⇒ `[1,2,1,2,1,3,1,3]`.

Câu hỏi: **khi nào có đáp án?** Số phần tử `m` lớn nhất không thể vượt quá `ceil(n / 2)`: nếu `m > ceil(n / 2)` thì giữa các bản sao phải có ít nhất `m - 1` phần tử khác, nên cần `n >= 2m - 1`, tức `m <= (n + 1) / 2`. Ngược lại, nếu `m <= ceil(n / 2)` thì luôn có đáp án (chứng minh ở mục 10 và kiểm bằng máy).

## 5. Cách 1 — Brute Force: quay lui

**Ý tưởng.** Xây dãy từng phần tử một: ở mỗi bước thử mọi giá trị còn dư mà **khác phần tử vừa đặt**; nếu bí thì quay lui. Đúng định nghĩa nên chắc chắn tìm được đáp án nếu tồn tại, nhưng có thể thử rất nhiều nhánh.

In [5]:
class TooManyNodes(Exception):
    pass

def backtrack(barcodes: List[int], cap: int = 10 ** 6):
    """Trả về (đáp án hoặc None, số nút đã duyệt). Ném TooManyNodes nếu vượt cap."""
    n = len(barcodes)
    cnt = Counter(barcodes)
    values = sorted(cnt)
    res = []
    nodes = 0

    def dfs() -> bool:
        nonlocal nodes
        nodes += 1
        if nodes > cap:
            raise TooManyNodes
        if len(res) == n:
            return True
        for v in values:
            if cnt[v] > 0 and (not res or res[-1] != v):
                cnt[v] -= 1
                res.append(v)
                if dfs():
                    return True
                res.pop()
                cnt[v] += 1
        return False

    ok = dfs()
    return (res[:] if ok else None), nodes

for barcodes in examples:
    res, nodes = backtrack(barcodes)
    print(f"quay lui({barcodes}) = {res} ({nodes} nút)", "hợp lệ" if is_valid(barcodes, res) else "SAI")
    assert is_valid(barcodes, res)

quay lui([1, 1, 1, 2, 2, 2]) = [1, 2, 1, 2, 1, 2] (7 nút) hợp lệ
quay lui([1, 1, 1, 1, 2, 2, 3, 3]) = [1, 2, 1, 2, 1, 3, 1, 3] (9 nút) hợp lệ


**Độ phức tạp và nút thắt.** Trường hợp xấu là dãy có nhiều giá trị khác nhau và một giá trị rất phổ biến (`ceil(n / 2)` lần) mang nhãn **lớn nhất**: quay lui thử các giá trị nhỏ trước, đặt chúng sớm rồi mới thấy phần tử phổ biến bị kẹt. Số nút tăng **theo hàm mũ**. Đo thử với `n = 2h - 1`: các giá trị `1..h-1` mỗi giá trị một lần và giá trị `99` xuất hiện `h` lần:

In [6]:
for h in range(2, 9):
    barcodes = list(range(1, h)) + [99] * h
    try:
        res, nodes = backtrack(barcodes, cap=3 * 10 ** 5)
        print(f"h={h} n={len(barcodes):2}: {nodes:8} nút", "hợp lệ" if is_valid(barcodes, res) else "SAI")
    except TooManyNodes:
        print(f"h={h} n={len(barcodes):2}: vượt 300000 nút, bỏ cuộc")

h=2 n= 3:        6 nút hợp lệ
h=3 n= 5:       20 nút hợp lệ
h=4 n= 7:      100 nút hợp lệ
h=5 n= 9:      734 nút hợp lệ
h=6 n=11:     7066 nút hợp lệ
h=7 n=13:    83040 nút hợp lệ


h=8 n=15: vượt 300000 nút, bỏ cuộc


Nút thắt: ta không biết **phải đặt giá trị nào trước** để khỏi bị kẹt. Câu trả lời: giá trị phổ biến nhất cần được ưu tiên ở các vị trí chẵn.

## 6. Ý tưởng thuật toán / bước đột phá

**Ý tưởng.** Chia `n` vị trí thành hai nhóm: các vị trí chẵn `0, 2, 4, ...` (có `h = ceil(n / 2)` chỗ) rồi các vị trí lẻ `1, 3, 5, ...` (có `n // 2` chỗ). Xếp các giá trị theo **thứ tự điền**: giá trị phổ biến nhất trước, mỗi giá trị chiếm một đoạn **liên tiếp** trong thứ tự điền. Ghi lần lượt vào các vị trí chẵn rồi các vị trí lẻ.

**Vì sao không đụng nhau?** Hai ô kề nhau trong dãy kết quả ứng với hai vị trí trong thứ tự điền cách nhau đúng `h` hoặc `h - 1` ô. Một giá trị chỉ chiếm một đoạn liên tiếp dài `c <= h`, nên hai bản sao của nó cách nhau tối đa `c - 1 <= h - 1`. Trường hợp khoảng cách bằng `h - 1` cần `c = h`; khi đó giá trị này phổ biến nhất nên đứng **đầu** thứ tự điền và nằm hết trong nửa chẵn, không có bản sao nào ở nửa lẻ. Chứng minh đầy đủ ở mục 10.

| Cách | Ý tưởng |
|---|---|
| 2 | Heap tham lam: mỗi bước lấy giá trị còn nhiều nhất khác giá trị vừa đặt |
| 3 | Đếm + sắp xếp giá trị theo tần suất giảm dần rồi điền chẵn-lẻ (nộp) |
| 4 | Chỉ cần đặt giá trị phổ biến nhất trước, các giá trị còn lại thứ tự bất kỳ |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Heap tham lam

**Giải thích từng đoạn.**

- Đưa `(-tần suất, giá trị)` của mọi giá trị vào max-heap.
- Mỗi bước lấy giá trị còn nhiều nhất, thêm vào kết quả, giảm tần suất đi 1.
- Không được đặt lại ngay giá trị vừa dùng, nên **giữ nó lại** (`held`) và chỉ đẩy vào heap ở bước sau, sau khi đã lấy phần tử mới.
- Nếu còn dư tần suất thì giá trị vừa dùng sẽ được đẩy lại ở bước kế tiếp.

In [7]:
def distant_heap(barcodes: List[int]) -> List[int]:
    cnt = Counter(barcodes)
    heap = [(-c, v) for v, c in cnt.items()]
    heapq.heapify(heap)
    res = []
    held = None                                    # (số âm tần suất còn lại, giá trị) vừa dùng ở bước trước
    while heap:
        c, v = heapq.heappop(heap)
        res.append(v)
        if held is not None:
            heapq.heappush(heap, held)             # bước trước đã dùng xong lượt cấm, trả lại heap
            held = None
        if c + 1 < 0:                              # còn dư (c là số âm: dùng một lần thì c + 1)
            held = (c + 1, v)
    return res

for barcodes in examples:
    res = distant_heap(barcodes)
    print(f"heap({barcodes}) = {res}", "hợp lệ" if is_valid(barcodes, res) else "SAI")
    assert is_valid(barcodes, res)

heap([1, 1, 1, 2, 2, 2]) = [1, 2, 1, 2, 1, 2] hợp lệ
heap([1, 1, 1, 1, 2, 2, 3, 3]) = [1, 2, 1, 3, 1, 2, 1, 3] hợp lệ


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
        import heapq
        from collections import Counter
        heap = [(-c, v) for v, c in Counter(barcodes).items()]   # max-heap theo tần suất
        heapq.heapify(heap)
        res = []
        held = None                                              # giá trị vừa dùng, chưa được dùng lại ngay
        while heap:
            c, v = heapq.heappop(heap)
            res.append(v)
            if held is not None:
                heapq.heappush(heap, held)
                held = None
            if c + 1 < 0:
                held = (c + 1, v)
        return res

sol_heap = lambda b: Solution().rearrangeBarcodes(b[:])
for barcodes in examples:
    assert is_valid(barcodes, sol_heap(barcodes))
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n log k)` với `k` là số giá trị khác nhau, bộ nhớ `O(k)` cho heap cùng `O(n)` cho kết quả.

---

### 7.2. Cách 3 — Đếm + sắp xếp theo tần suất + điền chẵn-lẻ (cách nộp chính)

**Giải thích từng đoạn.**

- `cnt = Counter(barcodes)`.
- `order`: các giá trị **khác nhau** sắp theo tần suất giảm dần (tie: theo giá trị tăng dần để ổn định).
- Với từng giá trị `v` theo `order`, ghi `cnt[v]` lần vào `res[i]`, mỗi lần `i += 2`; nếu `i >= n` thì chuyển sang vị trí lẻ: `i = 1`.

In [9]:
def distant_sorted_fill(barcodes: List[int]) -> List[int]:
    n = len(barcodes)
    cnt = Counter(barcodes)
    order = sorted(cnt, key=lambda x: (-cnt[x], x))
    res = [0] * n
    i = 0
    for v in order:
        for _ in range(cnt[v]):
            res[i] = v
            i += 2
            if i >= n:
                i = 1
    return res

for barcodes in examples:
    res = distant_sorted_fill(barcodes)
    print(f"sắp xếp + điền({barcodes}) = {res}", "hợp lệ" if is_valid(barcodes, res) else "SAI")
    assert is_valid(barcodes, res)

sắp xếp + điền([1, 1, 1, 2, 2, 2]) = [1, 2, 1, 2, 1, 2] hợp lệ
sắp xếp + điền([1, 1, 1, 1, 2, 2, 3, 3]) = [1, 2, 1, 2, 1, 3, 1, 3] hợp lệ


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
        from collections import Counter
        n = len(barcodes)
        cnt = Counter(barcodes)
        order = sorted(cnt, key=lambda x: (-cnt[x], x))   # giá trị phổ biến nhất trước
        res = [0] * n
        i = 0                                             # vị trí đang ghi: 0, 2, 4, ... rồi 1, 3, 5, ...
        for v in order:
            for _ in range(cnt[v]):
                res[i] = v
                i += 2
                if i >= n:
                    i = 1
        return res

sol_sorted = lambda b: Solution().rearrangeBarcodes(b[:])
for barcodes in examples:
    assert is_valid(barcodes, sol_sorted(barcodes))
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** đếm `O(n)`, sắp xếp `O(k log k)` với `k <= n` giá trị khác nhau, điền `O(n)`; tổng `O(n + k log k)`. Bộ nhớ `O(n)`.

---

### 7.3. Cách 4 — Chỉ cần đặt giá trị phổ biến nhất trước

**Ý tưởng khác.** Chứng minh ở mục 6 chỉ cần giá trị phổ biến nhất đứng **đầu** thứ tự điền; các giá trị còn lại thứ tự gì cũng được, miễn mỗi giá trị chiếm một đoạn liên tiếp. Do đó không cần sắp xếp: `max` tìm giá trị phổ biến nhất `O(k)`, rồi điền theo thứ tự `[mode] + các giá trị còn lại`. Tổng `O(n)`.

In [11]:
def distant_mode_first(barcodes: List[int]) -> List[int]:
    n = len(barcodes)
    cnt = Counter(barcodes)
    mode = max(cnt, key=cnt.get)
    order = [mode] + [v for v in cnt if v != mode]
    res = [0] * n
    i = 0
    for v in order:
        for _ in range(cnt[v]):
            res[i] = v
            i += 2
            if i >= n:
                i = 1
    return res

for barcodes in examples:
    res = distant_mode_first(barcodes)
    print(f"mode trước({barcodes}) = {res}", "hợp lệ" if is_valid(barcodes, res) else "SAI")
    assert is_valid(barcodes, res)

mode trước([1, 1, 1, 2, 2, 2]) = [1, 2, 1, 2, 1, 2] hợp lệ
mode trước([1, 1, 1, 1, 2, 2, 3, 3]) = [1, 2, 1, 2, 1, 3, 1, 3] hợp lệ


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
        from collections import Counter
        n = len(barcodes)
        cnt = Counter(barcodes)
        mode = max(cnt, key=cnt.get)                       # giá trị phổ biến nhất
        order = [mode] + [v for v in cnt if v != mode]     # chỉ cần mode đứng đầu
        res = [0] * n
        i = 0
        for v in order:
            for _ in range(cnt[v]):
                res[i] = v
                i += 2
                if i >= n:
                    i = 1
        return res

sol_mode = lambda b: Solution().rearrangeBarcodes(b[:])
for barcodes in examples:
    assert is_valid(barcodes, sol_mode(barcodes))
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Cách 3 cho `[1, 1, 1, 1, 2, 2, 3, 3]`

In thứ tự điền, từng lần ghi và lúc đổi sang vị trí lẻ.

In [13]:
def trace_fill(barcodes):
    n = len(barcodes)
    cnt = Counter(barcodes)
    order = sorted(cnt, key=lambda x: (-cnt[x], x))
    print("cnt =", dict(cnt), "| thứ tự điền =", order, "| n =", n)
    res = [0] * n
    i = 0
    for v in order:
        for _ in range(cnt[v]):
            res[i] = v
            print(f"  ghi {v} vào vị trí {i}: {res}")
            i += 2
            if i >= n:
                i = 1
                print("  hết vị trí chẵn -> chuyển sang vị trí lẻ (i = 1)")
    print("kết quả:", res, "hợp lệ" if is_valid(barcodes, res) else "SAI")
    return res

assert is_valid([1, 1, 1, 1, 2, 2, 3, 3], trace_fill([1, 1, 1, 1, 2, 2, 3, 3]))

cnt = {1: 4, 2: 2, 3: 2} | thứ tự điền = [1, 2, 3] | n = 8
  ghi 1 vào vị trí 0: [1, 0, 0, 0, 0, 0, 0, 0]
  ghi 1 vào vị trí 2: [1, 0, 1, 0, 0, 0, 0, 0]
  ghi 1 vào vị trí 4: [1, 0, 1, 0, 1, 0, 0, 0]
  ghi 1 vào vị trí 6: [1, 0, 1, 0, 1, 0, 1, 0]
  hết vị trí chẵn -> chuyển sang vị trí lẻ (i = 1)
  ghi 2 vào vị trí 1: [1, 2, 1, 0, 1, 0, 1, 0]
  ghi 2 vào vị trí 3: [1, 2, 1, 2, 1, 0, 1, 0]
  ghi 3 vào vị trí 5: [1, 2, 1, 2, 1, 3, 1, 0]
  ghi 3 vào vị trí 7: [1, 2, 1, 2, 1, 3, 1, 3]
  hết vị trí chẵn -> chuyển sang vị trí lẻ (i = 1)
kết quả: [1, 2, 1, 2, 1, 3, 1, 3] hợp lệ


### 8.2. Heap tham lam cho `[1, 1, 1, 2, 2, 3]`

In giá trị lấy ra, phần tử bị giữ lại và nội dung heap ở mỗi bước.

In [14]:
def trace_heap(barcodes):
    cnt = Counter(barcodes)
    heap = [(-c, v) for v, c in cnt.items()]
    heapq.heapify(heap)
    res, held = [], None
    step = 0
    while heap:
        c, v = heapq.heappop(heap)
        res.append(v)
        step += 1
        if held is not None:
            heapq.heappush(heap, held)
            held = None
        if c + 1 < 0:
            held = (c + 1, v)
        print(f"bước {step}: lấy {v} (còn {-(c + 1)}) | giữ lại {held} | heap = {sorted(heap)} | kết quả {res}")
    print("kết quả:", res, "hợp lệ" if is_valid(barcodes, res) else "SAI")
    return res

assert is_valid([1, 1, 1, 2, 2, 3], trace_heap([1, 1, 1, 2, 2, 3]))

bước 1: lấy 1 (còn 2) | giữ lại (-2, 1) | heap = [(-2, 2), (-1, 3)] | kết quả [1]
bước 2: lấy 2 (còn 1) | giữ lại (-1, 2) | heap = [(-2, 1), (-1, 3)] | kết quả [1, 2]
bước 3: lấy 1 (còn 1) | giữ lại (-1, 1) | heap = [(-1, 2), (-1, 3)] | kết quả [1, 2, 1]
bước 4: lấy 2 (còn 0) | giữ lại None | heap = [(-1, 1), (-1, 3)] | kết quả [1, 2, 1, 2]
bước 5: lấy 1 (còn 0) | giữ lại None | heap = [(-1, 3)] | kết quả [1, 2, 1, 2, 1]
bước 6: lấy 3 (còn 0) | giữ lại None | heap = [] | kết quả [1, 2, 1, 2, 1, 3]
kết quả: [1, 2, 1, 2, 1, 3] hợp lệ


### 8.3. Trường hợp biên: giá trị phổ biến nhất đúng `ceil(n / 2)` lần với `n` lẻ

`[2, 2, 2, 1, 3]`: `n = 5`, `h = 3`, số `2` phải chiếm **toàn bộ** các vị trí chẵn `0, 2, 4`.

In [15]:
res = trace_fill([2, 2, 2, 1, 3])
assert res[0::2] == [2, 2, 2]

cnt = {2: 3, 1: 1, 3: 1} | thứ tự điền = [2, 1, 3] | n = 5
  ghi 2 vào vị trí 0: [2, 0, 0, 0, 0]
  ghi 2 vào vị trí 2: [2, 0, 2, 0, 0]
  ghi 2 vào vị trí 4: [2, 0, 2, 0, 2]
  hết vị trí chẵn -> chuyển sang vị trí lẻ (i = 1)
  ghi 1 vào vị trí 1: [2, 1, 2, 0, 2]
  ghi 3 vào vị trí 3: [2, 1, 2, 3, 2]
  hết vị trí chẵn -> chuyển sang vị trí lẻ (i = 1)
kết quả: [2, 1, 2, 3, 2] hợp lệ


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (đếm + sắp xếp theo tần suất + điền chẵn-lẻ): `O(n + k log k)` thời gian, ngắn gọn và dễ chứng minh.

```python
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
        from collections import Counter
        n = len(barcodes)
        cnt = Counter(barcodes)
        order = sorted(cnt, key=lambda x: (-cnt[x], x))   # giá trị phổ biến nhất trước
        res = [0] * n
        i = 0                                             # vị trí đang ghi: 0, 2, 4, ... rồi 1, 3, 5, ...
        for v in order:
            for _ in range(cnt[v]):
                res[i] = v
                i += 2
                if i >= n:
                    i = 1
        return res
```

In [16]:
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
        from collections import Counter
        n = len(barcodes)
        cnt = Counter(barcodes)
        order = sorted(cnt, key=lambda x: (-cnt[x], x))   # giá trị phổ biến nhất trước
        res = [0] * n
        i = 0                                             # vị trí đang ghi: 0, 2, 4, ... rồi 1, 3, 5, ...
        for v in order:
            for _ in range(cnt[v]):
                res[i] = v
                i += 2
                if i >= n:
                    i = 1
        return res

sol = lambda b: Solution().rearrangeBarcodes(b[:])
for barcodes in examples:
    got = sol(barcodes)
    print(got, "hợp lệ" if is_valid(barcodes, got) else "SAI")
    assert is_valid(barcodes, got)

[1, 2, 1, 2, 1, 2] hợp lệ
[1, 2, 1, 2, 1, 3, 1, 3] hợp lệ


## 10. Vì sao đúng

**Điều kiện cần.** Gọi `m` là tần suất lớn nhất, `h = ceil(n / 2)`. Nếu `m > h` thì hai bản sao liên tiếp (theo vị trí) của giá trị đó có ít nhất `m - 1` khoảng trống ở giữa, mỗi khoảng chứa ít nhất một phần tử khác, nên cần `n >= m + (m - 1) = 2m - 1`, tức `m <= (n + 1) / 2`. Đề đảm bảo `m <= h`.

**Điều kiện đủ (chứng minh cho Cách 3 và 4).** Thứ tự điền `L` là dãy `n` phần tử, trong đó mỗi giá trị chiếm một **đoạn liên tiếp**. Đặt `h = ceil(n / 2)` (số vị trí chẵn). Phần tử thứ `p` của `L` (đếm từ `0`) được ghi vào vị trí `2p` nếu `p < h`, ngược lại vào vị trí `2(p - h) + 1`.

Hai vị trí kề nhau trong kết quả là một vị trí chẵn `2p` (`p < h`) và một vị trí lẻ `2r + 1` (ứng với `L` ở chỉ số `q = h + r`), với `2r + 1 = 2p + 1` hoặc `2r + 1 = 2p - 1`, tức `r = p` hoặc `r = p - 1`. Vậy `q - p = h` hoặc `q - p = h - 1`. Muốn hai vị trí này cùng giá trị, đoạn liên tiếp của giá trị đó phải chứa cả `p` và `q`, nên độ dài đoạn `c >= q - p + 1 >= h`. Vì `c <= m <= h`, chỉ có thể `c = h` và `q - p = h - 1`, tức `r = p - 1 >= 0` nên `p >= 1`.

- **Cách 3** (sắp xếp tần suất giảm dần): giá trị có tần suất `h` đứng ở đầu thứ tự điền. Nếu có hai giá trị cùng tần suất `h` thì `n = 2h` và chúng chiếm `L[0:h]` và `L[h:2h]`; nếu chỉ có một giá trị thì nó chiếm `L[0:h]`. Đoạn `L[0:h]` có `p` từ `0` đến `h - 1` và `q >= h`: không có `q` nào trong đoạn, mâu thuẫn. Đoạn `L[h:2h]` có `p >= h`, không thể là vị trí chẵn, mâu thuẫn.
- **Cách 4** (mode đứng đầu): chỉ giá trị có tần suất đúng `h` mới có thể vi phạm. Nếu đó là `mode` thì đoạn của nó là `L[0:h]`, không chứa `q >= h`. Nếu đó là một giá trị khác thì `mode` cũng có tần suất `h`, nên `n = 2h` và đoạn của giá trị kia là `L[h:2h]`, không chứa `p < h`. Mọi giá trị có tần suất `c <= h - 1` không thể vi phạm.

Vậy không có hai vị trí kề nhau cùng giá trị. Dãy kết quả là hoán vị của đầu vào (mỗi giá trị được ghi đúng `cnt[v]` lần) và `i` không bao giờ vượt quá vì ta ghi đúng `n` lần: các vị trí chẵn `h` lần, các vị trí lẻ `n - h` lần.

**Cách 2 (heap).** Mỗi bước lấy giá trị còn nhiều nhất trong những giá trị **khác giá trị vừa đặt** (vì giá trị vừa đặt đang bị giữ ngoài heap). Nếu còn `r` phần tử cần xếp và giá trị nhiều nhất có tần suất `<= ceil(r / 2)` thì invariant được giữ; kiểm bằng máy với oracle và hàm kiểm tra.

**Kiểm chứng bằng máy.** Với oracle độc lập: (1) liệt kê **mọi** hoán vị cho `n <= 7` để xác nhận định lý "có đáp án khi và chỉ khi `m <= ceil(n / 2)`"; (2) chạy mọi cách trên **mọi** đa tập có đáp án với `n <= 9` và giá trị `1..4`, kiểm bằng hàm `is_valid`.

In [17]:
# (1) định lý tồn tại: kiểm bằng cách liệt kê mọi hoán vị
def exists_bruteforce(barcodes):
    return any(all(p[k] != p[k + 1] for k in range(len(p) - 1)) for p in set(permutations(barcodes)))

checked = 0
for n in range(1, 8):
    for tup in product(range(1, 4), repeat=n):
        if list(tup) != sorted(tup):                  # chỉ cần mỗi đa tập một lần
            continue
        b = list(tup)
        m = max(Counter(b).values())
        assert exists_bruteforce(b) == (m <= (n + 1) // 2), b
        checked += 1
print("định lý 'có đáp án <=> m <= ceil(n / 2)' đúng trên", checked, "đa tập (n <= 7, giá trị 1..3)")

định lý 'có đáp án <=> m <= ceil(n / 2)' đúng trên 119 đa tập (n <= 7, giá trị 1..3)


In [18]:
# (2) mọi cách trên mọi đa tập có đáp án
all_impls = {"quay lui": lambda b: backtrack(b)[0], "heap": distant_heap, "sắp xếp + điền": distant_sorted_fill,
             "mode trước": distant_mode_first, "nộp (Cách 3)": sol, "Cách 2 (nộp)": sol_heap, "Cách 4 (nộp)": sol_mode}
total = 0
for n in range(1, 10):
    for tup in product(range(1, 5), repeat=n):
        if list(tup) != sorted(tup):
            continue
        b = list(tup)
        if max(Counter(b).values()) > (n + 1) // 2:
            continue
        for name, fn in all_impls.items():
            res = fn(b[:])
            assert is_valid(b, res), (name, b, res)
        total += 1
print("đã kiểm", total, "đa tập hợp lệ (n <= 9, giá trị 1..4): mọi cách cho đáp án hợp lệ")

đã kiểm 434 đa tập hợp lệ (n <= 9, giá trị 1..4): mọi cách cho đáp án hợp lệ


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | mũ (trường hợp xấu) | `O(n)` |
| 2. Heap tham lam | `O(n log k)` | `O(n)` |
| 3. Sắp xếp theo tần suất + điền | `O(n + k log k)` | `O(n)` |
| 4. Mode trước | `O(n)` | `O(n)` |

(`k` = số giá trị khác nhau, `k <= n`.) Đo ở cỡ tối đa (`n = 10^4`) với các dạng: ngẫu nhiên, mode đúng một nửa, `n` lẻ với mode `ceil(n / 2)`, hai giá trị, toàn khác nhau:

In [19]:
def make_valid(rng, n, mode_count, k):
    """Tạo dãy n phần tử, giá trị mode xuất hiện mode_count lần, các giá trị khác chia đều trong k - 1 giá trị."""
    barcodes = [1] * mode_count
    k_others = max(1, k - 1, -(-(n - mode_count) // mode_count))      # đủ nhiều giá trị khác để không giá trị nào vượt mode_count
    others = list(range(2, 2 + k_others))
    barcodes += [others[j % len(others)] for j in range(n - mode_count)]
    rng.shuffle(barcodes)
    return barcodes

rng = random.Random(11)
n = 10 ** 4
cases = {
    "ngẫu nhiên (k ~ 6000)": [rng.randint(1, 6000) for _ in range(n)],
    "mode đúng 5000 (n chẵn)": make_valid(rng, n, 5000, 50),
    "n lẻ, mode 5000": make_valid(rng, n - 1, 5000, 3),
    "hai giá trị 5000/5000": make_valid(rng, n, 5000, 2),
    "toàn khác nhau": rng.sample(range(1, n + 1), n),
    "một phần tử": [7],
}
for label, b in cases.items():
    assert max(Counter(b).values()) <= (len(b) + 1) // 2, label
    row = [f"{label:26}"]
    for name, fn in (("heap", sol_heap), ("sắp xếp", sol_sorted), ("mode", sol_mode), ("nộp", sol)):
        t0 = time.perf_counter()
        got = fn(b)
        dt = (time.perf_counter() - t0) * 1000
        assert is_valid(b, got), (label, name)
        row.append(f"{name} {dt:6.2f} ms")
    print(" | ".join(row))

ngẫu nhiên (k ~ 6000)      | heap   3.99 ms | sắp xếp   3.49 ms | mode   3.78 ms | nộp   3.27 ms
mode đúng 5000 (n chẵn)    | heap   0.88 ms | sắp xếp   0.80 ms | mode   0.80 ms | nộp   0.79 ms
n lẻ, mode 5000            | heap   0.78 ms | sắp xếp   0.76 ms | mode   0.79 ms | nộp   0.77 ms
hai giá trị 5000/5000      | heap   0.75 ms | sắp xếp   0.71 ms | mode   0.72 ms | nộp   0.82 ms
toàn khác nhau             | heap   7.01 ms | sắp xếp   6.08 ms | mode   5.50 ms | nộp   5.77 ms
một phần tử                | heap   0.02 ms | sắp xếp   0.00 ms | mode   0.00 ms | nộp   0.00 ms


## 12. Bộ test

Vì bài có **nhiều đáp án đúng**, mọi test dùng `is_valid` (cùng bội số phần tử, không có hai phần tử kề bằng nhau) chứ không so sánh với một đáp án cố định. Bộ test gồm: ví dụ của đề, edge case, kiểm tra `is_valid` từ chối kết quả sai, phản ví dụ cho các cách sai (tìm bằng tìm kiếm), test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [20]:
def check_all(barcodes, brute=True):
    assert max(Counter(barcodes).values()) <= (len(barcodes) + 1) // 2, "đầu vào vi phạm đảm bảo của đề"
    for name, fn in (("heap", sol_heap), ("sắp xếp", sol_sorted), ("mode", sol_mode), ("nộp", sol)):
        before = barcodes[:]
        res = fn(barcodes)
        assert is_valid(barcodes, res), (name, barcodes[:12], res[:12])
        assert barcodes == before, "đầu vào bị thay đổi"
    if brute and len(barcodes) <= 9:
        res, _ = backtrack(barcodes)
        assert is_valid(barcodes, res), barcodes

# 12.1 ví dụ đề
for barcodes in examples:
    check_all(barcodes)
print("12.1 ví dụ đề: OK")

# 12.2 edge case
edge = [
    [5],                            # một phần tử
    [1, 2],                         # hai giá trị khác nhau
    [3, 3, 4],                      # n lẻ, mode đúng ceil(n / 2) = 2
    [1, 1, 2, 2],                   # n chẵn, hai giá trị mỗi giá trị n / 2
    [2, 2, 2, 1, 3],                # n lẻ, mode = 3 = ceil(5 / 2): chiếm toàn bộ vị trí chẵn
    [7, 7, 7, 7, 1, 2, 3, 4],       # mode đúng n / 2
    [1, 2, 3, 4, 5],                # toàn khác nhau
    [10000, 10000, 1],              # biên giá trị
    [1, 1, 2, 2, 3, 3, 4, 4],       # nhiều giá trị cùng tần suất
    [4, 4, 4, 2, 2, 3, 3],          # mode ở giữa thứ tự đầu vào
    [1, 2, 2, 3, 3, 3],             # n chẵn, mode = n / 2
]
for b in edge:
    check_all(b)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 is_valid phải từ chối kết quả sai
assert not is_valid([1, 1, 2], [1, 1, 2])             # hai số 1 kề nhau
assert not is_valid([1, 1, 2], [1, 2, 2])             # không phải hoán vị
assert not is_valid([1, 1, 2], [1, 2])                # thiếu phần tử
assert not is_valid([1, 1, 2], [1, 2, 1, 0])          # thừa phần tử
assert is_valid([1, 1, 2], [1, 2, 1])
print("12.3 hàm kiểm tra: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 11 ca
12.3 hàm kiểm tra: OK


In [21]:
# 12.4 phản ví dụ: các cách sai (tìm bằng tìm kiếm trên các đầu vào hợp lệ)
def wrong_greedy_smallest(barcodes):               # tham lam: mỗi bước lấy giá trị nhỏ nhất khác giá trị trước
    cnt = Counter(barcodes)
    res = []
    for _ in range(len(barcodes)):
        cands = [v for v in sorted(cnt) if cnt[v] > 0 and (not res or res[-1] != v)]
        v = cands[0] if cands else res[-1]         # bí: đành lặp lại
        res.append(v)
        cnt[v] -= 1 if cnt[v] > 0 else 0
    return res

def wrong_no_frequency_sort(barcodes):             # điền chẵn-lẻ nhưng sắp theo giá trị, bỏ qua tần suất
    n = len(barcodes)
    cnt = Counter(barcodes)
    res = [0] * n
    i = 0
    for v in sorted(cnt):
        for _ in range(cnt[v]):
            res[i] = v
            i += 2
            if i >= n:
                i = 1
    return res

def wrong_sort_not_grouped(barcodes):              # sắp theo tần suất nhưng giá trị cùng tần suất bị xen kẽ (sắp ổn định)
    n = len(barcodes)
    cnt = Counter(barcodes)
    order = sorted(barcodes, key=lambda x: -cnt[x])
    res = [0] * n
    i = 0
    for v in order:
        res[i] = v
        i += 2
        if i >= n:
            i = 1
    return res

def wrong_odd_first(barcodes):                     # bắt đầu từ vị trí lẻ
    n = len(barcodes)
    cnt = Counter(barcodes)
    order = sorted(cnt, key=lambda x: (-cnt[x], x))
    res = [0] * n
    i = 1 if n > 1 else 0
    for v in order:
        for _ in range(cnt[v]):
            res[i] = v
            i += 2
            if i >= n:
                i = 0
    return res

def wrong_heap_no_hold(barcodes):                  # heap nhưng đẩy lại giá trị vừa dùng ngay lập tức
    cnt = Counter(barcodes)
    heap = [(-c, v) for v, c in cnt.items()]
    heapq.heapify(heap)
    res = []
    while heap:
        c, v = heapq.heappop(heap)
        res.append(v)
        if c + 1 < 0:
            heapq.heappush(heap, (c + 1, v))
    return res

def find_counter(wrong, tries=50000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 9)
        b = [rng.randint(1, 4) for _ in range(n)]
        if max(Counter(b).values()) > (n + 1) // 2:
            continue
        try:
            res = wrong(b)
        except Exception:
            return b
        if not is_valid(b, res):
            return b
    return None

counters = {}
for name, fn in (("tham lam giá trị nhỏ nhất", wrong_greedy_smallest), ("không sắp theo tần suất", wrong_no_frequency_sort),
                 ("sắp theo tần suất nhưng không gom giá trị", wrong_sort_not_grouped), ("điền vị trí lẻ trước", wrong_odd_first),
                 ("heap không giữ lại giá trị vừa dùng", wrong_heap_no_hold)):
    b = find_counter(fn)
    assert b is not None, name
    counters[name] = b
    print(f"{name:42} barcodes={b}: sai -> {fn(b)}")
    check_all(b)
print("12.4 phản ví dụ: OK")

tham lam giá trị nhỏ nhất                  barcodes=[3, 3, 2, 2, 1, 3, 4, 1, 3]: sai -> [1, 2, 1, 2, 3, 4, 3, 3, 3]
không sắp theo tần suất                    barcodes=[4, 3, 3, 1, 3]: sai -> [1, 3, 3, 4, 3]
sắp theo tần suất nhưng không gom giá trị  barcodes=[2, 3, 1, 1, 2]: sai -> [2, 2, 1, 3, 1]
điền vị trí lẻ trước                       barcodes=[3, 2, 2]: sai -> [2, 2, 3]
heap không giữ lại giá trị vừa dùng        barcodes=[3, 3, 2, 2, 1, 3, 4, 1, 3]: sai -> [3, 3, 1, 2, 3, 1, 2, 3, 4]
12.4 phản ví dụ: OK


In [22]:
# 12.5 ngẫu nhiên nhỏ: mọi dải giá trị, mode có thể đúng ceil(n / 2)
rng = random.Random(4054)
cnt_tests = 0
for _ in range(3000):
    n = rng.randint(1, 14)
    hi = rng.choice([1, 2, 3, 5, 10, 10 ** 4])
    b = [rng.randint(1, hi) for _ in range(n)]
    if max(Counter(b).values()) > (n + 1) // 2:
        continue
    check_all(b, brute=n <= 9)
    cnt_tests += 1
print("12.5 ngẫu nhiên nhỏ: OK", cnt_tests, "ca")

# 12.6 ngẫu nhiên vừa và lớn: dựng đầu vào hợp lệ bằng cách chọn tần suất mode <= ceil(n / 2)
for _ in range(60):
    n = rng.randint(20, 400)
    m = rng.randint(1, (n + 1) // 2)
    check_all(make_valid(rng, n, m, rng.randint(2, 30)), brute=False)
for _ in range(15):
    n = rng.randint(5000, 10 ** 4)
    m = rng.choice([(n + 1) // 2, (n + 1) // 2 - 1, rng.randint(1, (n + 1) // 2)])
    check_all(make_valid(rng, n, m, rng.choice([2, 3, 50, 3000])), brute=False)
print("12.6 ngẫu nhiên vừa và lớn: OK")

# 12.7 tính chất: kết quả không phụ thuộc thứ tự đầu vào về mặt hợp lệ; trộn đầu vào cho kết quả vẫn hợp lệ
for _ in range(300):
    n = rng.randint(1, 30)
    m = rng.randint(1, (n + 1) // 2)
    b = make_valid(rng, n, m, rng.randint(2, 6))
    shuffled = b[:]
    rng.shuffle(shuffled)
    assert is_valid(b, sol(shuffled)) and is_valid(shuffled, sol(b))
    assert sol(b) == sol(shuffled)                  # Cách 3 sắp xếp theo (tần suất, giá trị): kết quả không phụ thuộc thứ tự đầu vào
print("12.7 tính chất: OK")

12.5 ngẫu nhiên nhỏ: OK 2138 ca


12.6 ngẫu nhiên vừa và lớn: OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: mỗi bước lấy giá trị nhỏ nhất khác giá trị vừa đặt

Cách này không nhìn xa: nó dùng hết các giá trị nhỏ trước rồi để lại giá trị phổ biến đứng cạnh nhau.

In [23]:
b = counters["tham lam giá trị nhỏ nhất"]
print("barcodes =", b, "| sai ->", wrong_greedy_smallest(b), "| hợp lệ?", is_valid(b, wrong_greedy_smallest(b)), "| đúng ->", sol(b))
assert not is_valid(b, wrong_greedy_smallest(b)) and is_valid(b, sol(b))

barcodes = [3, 3, 2, 2, 1, 3, 4, 1, 3] | sai -> [1, 2, 1, 2, 3, 4, 3, 3, 3] | hợp lệ? False | đúng -> [3, 1, 3, 2, 3, 2, 3, 4, 1]


Vì sao sai: quyết định cục bộ "nhỏ nhất" bỏ qua việc giá trị phổ biến cần nhiều chỗ nhất. **Đúng:** ưu tiên giá trị nhiều nhất (heap) hoặc xếp theo tần suất rồi rải chẵn-lẻ.

### Lỗi 2 — Điền chẵn-lẻ nhưng không sắp xếp theo tần suất

In [24]:
b = counters["không sắp theo tần suất"]
print("barcodes =", b, "| sai ->", wrong_no_frequency_sort(b), "| hợp lệ?", is_valid(b, wrong_no_frequency_sort(b)), "| đúng ->", sol(b))
assert not is_valid(b, wrong_no_frequency_sort(b)) and is_valid(b, sol(b))

barcodes = [4, 3, 3, 1, 3] | sai -> [1, 3, 3, 4, 3] | hợp lệ? False | đúng -> [3, 1, 3, 4, 3]


Vì sao sai: nếu giá trị phổ biến nhất không đứng đầu thứ tự điền thì nó có thể trải qua ranh giới giữa nửa chẵn và nửa lẻ và hai bản sao rơi vào hai ô kề nhau. **Đúng:** giá trị phổ biến nhất đứng **đầu** thứ tự điền.

### Lỗi 3 — Sắp theo tần suất nhưng không gom các giá trị cùng tần suất

`sorted(barcodes, key=lambda x: -cnt[x])` giữ nguyên thứ tự đầu vào của các phần tử cùng tần suất, nên các giá trị cùng tần suất bị xen kẽ nhau và mất tính "một giá trị chiếm một đoạn liên tiếp".

In [25]:
b = counters["sắp theo tần suất nhưng không gom giá trị"]
print("barcodes =", b, "| sai ->", wrong_sort_not_grouped(b), "| hợp lệ?", is_valid(b, wrong_sort_not_grouped(b)), "| đúng ->", sol(b))
assert not is_valid(b, wrong_sort_not_grouped(b)) and is_valid(b, sol(b))

barcodes = [2, 3, 1, 1, 2] | sai -> [2, 2, 1, 3, 1] | hợp lệ? False | đúng -> [1, 2, 1, 3, 2]


Vì sao sai: chứng minh dựa vào việc mỗi giá trị là một đoạn liên tiếp trong thứ tự điền. **Đúng:** khóa sắp xếp phải có thêm **giá trị** làm tiêu chí phụ: `key=lambda x: (-cnt[x], x)`, hoặc sắp xếp các giá trị khác nhau rồi lặp lại `cnt[v]` lần.

### Lỗi 4 — Điền vị trí lẻ trước

Có `ceil(n / 2)` vị trí chẵn nhưng chỉ có `n // 2` vị trí lẻ; giá trị phổ biến nhất có thể cần cả `ceil(n / 2)` vị trí không kề nhau, chỉ vị trí chẵn đủ chỗ khi `n` lẻ.

In [26]:
b = counters["điền vị trí lẻ trước"]
print("barcodes =", b, "| sai ->", wrong_odd_first(b), "| hợp lệ?", is_valid(b, wrong_odd_first(b)), "| đúng ->", sol(b))
assert not is_valid(b, wrong_odd_first(b)) and is_valid(b, sol(b))

barcodes = [3, 2, 2] | sai -> [2, 2, 3] | hợp lệ? False | đúng -> [2, 3, 2]


Vì sao sai: giá trị phổ biến nhất tràn sang vị trí chẵn và đụng nhau. **Đúng:** bắt đầu từ vị trí chẵn `0`.

### Lỗi 5 — Heap quên "giữ lại" giá trị vừa dùng

Nếu đẩy lại giá trị vừa dùng vào heap ngay thì bước sau có thể lấy lại chính nó (vì nó vẫn đang nhiều nhất), tạo hai phần tử bằng nhau cạnh nhau.

In [27]:
b = counters["heap không giữ lại giá trị vừa dùng"]
print("barcodes =", b, "| sai ->", wrong_heap_no_hold(b), "| hợp lệ?", is_valid(b, wrong_heap_no_hold(b)), "| đúng ->", sol_heap(b))
assert not is_valid(b, wrong_heap_no_hold(b)) and is_valid(b, sol_heap(b))

barcodes = [3, 3, 2, 2, 1, 3, 4, 1, 3] | sai -> [3, 3, 1, 2, 3, 1, 2, 3, 4] | hợp lệ? False | đúng -> [3, 1, 3, 2, 3, 2, 3, 4, 1]


Vì sao sai: giá trị vừa đặt không được đặt lại ở bước kế tiếp. **Đúng:** lấy giá trị mới **trước**, rồi mới trả giá trị bị giữ về heap.

### Lỗi 6 — Dùng `n // 2` thay vì `(n + 1) // 2` khi chia đôi

Khi viết bằng gán lát cắt, nửa chẵn có `(n + 1) // 2` ô; nếu chia theo `n // 2` với `n` lẻ thì độ dài hai vế không khớp.

In [28]:
def wrong_floor_half(barcodes):
    n = len(barcodes)
    cnt = Counter(barcodes)
    order = [v for v in sorted(cnt, key=lambda x: (-cnt[x], x)) for _ in range(cnt[v])]
    res = [0] * n
    res[0::2] = order[:n // 2]               # SAI: nửa chẵn có (n + 1) // 2 ô
    res[1::2] = order[n // 2:]
    return res

try:
    wrong_floor_half([1, 1, 1, 2, 3])
    print("không có lỗi (không mong đợi)")
except ValueError as e:
    print("ValueError:", e)

ValueError: attempt to assign sequence of size 2 to extended slice of size 3


Vì sao sai: `res[0::2]` có `ceil(n / 2)` ô nên cần `order[:(n + 1) // 2]`. **Đúng:** `res[0::2] = order[:(n + 1) // 2]` và `res[1::2] = order[(n + 1) // 2:]`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "sắp xếp lại để không có hai phần tử kề giống nhau / cách nhau ít nhất `k`" ⇒ **tham lam theo tần suất**: xử lý giá trị nhiều nhất trước.
- Hai cách kinh điển: **heap** (mỗi bước lấy giá trị nhiều nhất không bị cấm) và **rải chẵn-lẻ** theo thứ tự tần suất.
- Điều kiện tồn tại: `m <= ceil(n / 2)`.
- Khi nhiều đáp án đúng, test bằng **hàm kiểm tra hợp lệ** chứ không so sánh với một đáp án mẫu.
- Dạng viết gọn bằng gán lát cắt: `res[0::2] = order[:(n + 1) // 2]; res[1::2] = order[(n + 1) // 2:]` (với `order` là dãy đã sắp xếp theo tần suất, gom các giá trị trùng).
- Cùng họ: Reorganize String (767), Task Scheduler (621), Rearrange String k Distance Apart (358, Premium).
- Test nên có: `n = 1`, mode đúng `ceil(n / 2)` với `n` lẻ và chẵn, nhiều giá trị cùng tần suất, toàn khác nhau.

## 15. Biến thể

1. **Reorganize String (767):** cùng bài với ký tự; nếu không thể thì trả về chuỗi rỗng.
2. **Hoán vị hợp lệ nhỏ nhất theo thứ tự từ điển** (thay vì "bất kỳ").
3. **Khoảng cách tối thiểu `k`** (358, 621): heap + hàng đợi chờ.
4. Bài liên quan: 358, 621, 767.

Kiểm biến thể 1 bằng oracle liệt kê mọi hoán vị:

In [29]:
def reorganize_string(s: str) -> str:
    arr = [ord(ch) for ch in s]
    n = len(arr)
    if n and max(Counter(arr).values()) > (n + 1) // 2:
        return ""
    return "".join(chr(x) for x in distant_sorted_fill(arr))

def reorganize_valid(s: str, t: str) -> bool:
    if t == "":
        return not any(all(p[k] != p[k + 1] for k in range(len(p) - 1)) for p in set(permutations(s)))
    return sorted(t) == sorted(s) and all(t[k] != t[k + 1] for k in range(len(t) - 1))

assert reorganize_string("aab") in ("aba",) and reorganize_string("aaab") == ""
rng = random.Random(15)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    assert reorganize_valid(s, reorganize_string(s)), s
print("biến thể Reorganize String: kiểm bằng oracle hoán vị")

# Biến thể 2: hoán vị hợp lệ nhỏ nhất theo thứ tự từ điển, tham lam với kiểm tra khả thi
def smallest_valid(barcodes: List[int]):
    cnt = Counter(barcodes)
    n = len(barcodes)
    res = []

    def feasible(r, prev):
        # còn r phần tử, phần tử đầu phải khác prev: tồn tại hoán vị hợp lệ?
        if r == 0:
            return True
        m = max(c for c in cnt.values())
        half = (r + 1) // 2
        if m > half:
            return False
        if r % 2 == 1 and prev is not None and cnt.get(prev, 0) == half:
            return False                                  # giá trị này buộc phải đứng đầu
        return True

    for pos in range(n):
        for v in sorted(cnt):
            if cnt[v] > 0 and (not res or res[-1] != v):
                cnt[v] -= 1
                if feasible(n - pos - 1, v):
                    res.append(v)
                    break
                cnt[v] += 1
        else:
            return None
    return res

def smallest_valid_brute(barcodes):
    good = [list(p) for p in set(permutations(barcodes)) if all(p[k] != p[k + 1] for k in range(len(p) - 1))]
    return min(good) if good else None

for _ in range(800):
    b = [rng.randint(1, 4) for _ in range(rng.randint(1, 7))]
    assert smallest_valid(b) == smallest_valid_brute(b), b
print("biến thể 'nhỏ nhất theo từ điển': tham lam + kiểm tra khả thi khớp oracle")

biến thể Reorganize String: kiểm bằng oracle hoán vị


biến thể 'nhỏ nhất theo từ điển': tham lam + kiểm tra khả thi khớp oracle


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | Thử mọi khả năng | mũ | `O(n)` | Chỉ để kiểm tra dữ liệu rất nhỏ |
| 2. Heap tham lam | Lấy giá trị nhiều nhất không bị cấm | `O(n log k)` | `O(n)` | Dễ mở rộng cho khoảng cách `k` |
| 3. Sắp xếp + điền chẵn-lẻ | Giá trị đoạn liên tiếp, rải chẵn rồi lẻ | `O(n + k log k)` | `O(n)` | **Cách nộp chính** |
| 4. Mode trước | Chỉ cần mode đứng đầu | `O(n)` | `O(n)` | Nhanh nhất, ít sắp xếp |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
cnt = Counter([3, 1, 3, 2, 3, 1])
print(sorted(cnt, key=lambda x: (-cnt[x], x)))
print(Solution().rearrangeBarcodes([3, 1, 3, 2, 3, 1]))
```

In [30]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # thứ tự các giá trị
my_guess_2 = None      # kết quả của Solution (cách nộp)

### Bài 2 — Tính tay (dễ-vừa)

`barcodes = [2, 2, 2, 3, 3, 1, 1]`. Theo Cách 3 (sắp theo tần suất giảm dần, hòa thì theo giá trị tăng dần), ghi kết quả. Kiểm tra bằng mắt rằng không có hai số kề nhau bằng nhau.

In [31]:
# Bài 2: điền kết quả tính tay
hand_result = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành phần điền chẵn-lẻ.

In [32]:
def exercise_3(barcodes: List[int]) -> List[int]:
    n = len(barcodes)
    cnt = Counter(barcodes)
    order = sorted(cnt, key=lambda x: (-cnt[x], x))
    res = [0] * n
    i = 0
    for v in order:
        for _ in range(cnt[v]):
            # TODO 1: ghi v vào res[i]
            # TODO 2: tăng i thêm 2
            # TODO 3: nếu i >= n thì i = 1
            pass
    return res


# for b in examples:
#     assert is_valid(b, exercise_3(b))

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, tự viết `exercise_4(barcodes)` dùng **heap** (nhớ giữ lại giá trị vừa dùng).

In [33]:
def exercise_4(barcodes: List[int]) -> List[int]:
    # TODO
    pass


# for b in examples:
#     assert is_valid(b, exercise_4(b))

### Bài 5 — Mini challenge: Reorganize String (khó)

Cho chuỗi `s` gồm chữ thường. Viết `exercise_5(s)` trả về **một** chuỗi hoán vị của `s` sao cho không có hai ký tự kề nhau bằng nhau, hoặc `""` nếu không thể. Có nhiều đáp án đúng nên kiểm bằng hàm kiểm tra.

```python
exercise_5("aab")   -> "aba"
exercise_5("aaab")  -> ""
exercise_5("a")     -> "a"
```
Gợi ý: kiểm tra điều kiện tồn tại `m <= (n + 1) // 2` rồi dùng cách rải chẵn-lẻ.

In [34]:
def exercise_5(s: str) -> str:
    # TODO
    pass


def check_reorganize(s: str, t: str) -> bool:
    n = len(s)
    possible = n == 0 or max(Counter(s).values()) <= (n + 1) // 2
    if not possible:
        return t == ""
    return sorted(t) == sorted(s) and all(t[k] != t[k + 1] for k in range(len(t) - 1))


# for s in ["aab", "aaab", "a", "aabb", "abcabc", "aaabbbc"]:
#     assert check_reorganize(s, exercise_5(s)), s

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `Counter` đếm: `3` xuất hiện `3` lần, `1` hai lần, `2` một lần.
- Mức 2: thứ tự điền `[3, 1, 2]`; `n = 6`: vị trí chẵn `0, 2, 4` nhận ba số `3`.
- Mức 3: dòng 1: `[3, 1, 2]`; dòng 2: `[3, 1, 3, 1, 3, 2]` (vị trí lẻ `1, 3, 5` nhận `1, 1, 2`).

**Bài 2**
- Mức 1: tần suất: `2` ba lần, `3` hai lần, `1` hai lần; thứ tự điền `[2, 1, 3]`.
- Mức 2: `n = 7`, vị trí chẵn `0, 2, 4, 6` nhận `2, 2, 2, 1`; vị trí lẻ `1, 3, 5` nhận `1, 3, 3`.
- Mức 3: `[2, 1, 2, 3, 2, 3, 1]`.

**Bài 3**
- Mức 1: `res[i] = v`.
- Mức 2: `i += 2`.
- Mức 3: `if i >= n: i = 1`.

**Bài 4**
- Mức 1: heap chứa `(-tần suất, giá trị)`.
- Mức 2: mỗi bước `heappop`, thêm vào kết quả, rồi đẩy lại **giá trị bị giữ ở bước trước** (nếu có).
- Mức 3: giữ lại `(c + 1, v)` nếu `c + 1 < 0`.

**Bài 5**
- Mức 1: đếm ký tự, kiểm `max count <= (n + 1) // 2`.
- Mức 2: sắp ký tự theo (tần suất giảm, ký tự), điền chẵn rồi lẻ.
- Mức 3: ghép kết quả bằng `"".join(res)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [35]:
cnt = Counter([3, 1, 3, 2, 3, 1])
r1 = sorted(cnt, key=lambda x: (-cnt[x], x))
r2 = Solution().rearrangeBarcodes([3, 1, 3, 2, 3, 1])
print(r1)
print(r2)
assert r1 == [3, 1, 2]
assert r2 == [3, 1, 3, 1, 3, 2]
assert is_valid([3, 1, 3, 2, 3, 1], r2)

[3, 1, 2]
[3, 1, 3, 1, 3, 2]


### Đáp án Bài 2

Thứ tự điền `[2, 1, 3]`; các giá trị xếp thành `L = [2, 2, 2, 1, 1, 3, 3]`; vị trí chẵn `0, 2, 4, 6` nhận `2, 2, 2, 1`; vị trí lẻ `1, 3, 5` nhận `1, 3, 3`. Kết quả `[2, 1, 2, 3, 2, 3, 1]`.

In [36]:
hand_result_solution = [2, 1, 2, 3, 2, 3, 1]
b = [2, 2, 2, 3, 3, 1, 1]
assert is_valid(b, hand_result_solution)
assert hand_result_solution == sol(b)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [37]:
def exercise_3_solution(barcodes: List[int]) -> List[int]:
    n = len(barcodes)
    cnt = Counter(barcodes)
    order = sorted(cnt, key=lambda x: (-cnt[x], x))
    res = [0] * n
    i = 0
    for v in order:
        for _ in range(cnt[v]):
            res[i] = v
            i += 2
            if i >= n:
                i = 1
    return res

for b in examples:
    assert is_valid(b, exercise_3_solution(b))
rng = random.Random(3)
for _ in range(500):
    n = rng.randint(1, 20)
    m = rng.randint(1, (n + 1) // 2)
    b = make_valid(rng, n, m, rng.randint(2, 6))
    assert is_valid(b, exercise_3_solution(b)), b
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [38]:
def exercise_4_solution(barcodes: List[int]) -> List[int]:
    heap = [(-c, v) for v, c in Counter(barcodes).items()]
    heapq.heapify(heap)
    res, held = [], None
    while heap:
        c, v = heapq.heappop(heap)
        res.append(v)
        if held is not None:
            heapq.heappush(heap, held)
            held = None
        if c + 1 < 0:
            held = (c + 1, v)
    return res

for b in examples:
    assert is_valid(b, exercise_4_solution(b))
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 20)
    m = rng.randint(1, (n + 1) // 2)
    b = make_valid(rng, n, m, rng.randint(2, 6))
    assert is_valid(b, exercise_4_solution(b)), b
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [39]:
def exercise_5_solution(s: str) -> str:
    n = len(s)
    cnt = Counter(s)
    if n and max(cnt.values()) > (n + 1) // 2:
        return ""
    order = sorted(cnt, key=lambda x: (-cnt[x], x))
    res = [""] * n
    i = 0
    for ch in order:
        for _ in range(cnt[ch]):
            res[i] = ch
            i += 2
            if i >= n:
                i = 1
    return "".join(res)

def check_reorganize_solution(s, t):
    n = len(s)
    possible = n == 0 or max(Counter(s).values()) <= (n + 1) // 2
    if not possible:
        return t == ""
    return sorted(t) == sorted(s) and all(t[k] != t[k + 1] for k in range(len(t) - 1))

for s in ["aab", "aaab", "a", "aabb", "abcabc", "aaabbbc"]:
    assert check_reorganize_solution(s, exercise_5_solution(s)), s
assert exercise_5_solution("aaab") == "" and exercise_5_solution("a") == "a"
rng = random.Random(5)
for _ in range(500):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 8)))
    t = exercise_5_solution(s)
    assert check_reorganize_solution(s, t) and reorganize_valid(s, t), s
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (đếm + sắp xếp theo tần suất + điền chẵn-lẻ)

```python
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
        from collections import Counter
        n = len(barcodes)
        cnt = Counter(barcodes)
        order = sorted(cnt, key=lambda x: (-cnt[x], x))   # giá trị phổ biến nhất trước
        res = [0] * n
        i = 0                                             # 0, 2, 4, ... rồi 1, 3, 5, ...
        for v in order:
            for _ in range(cnt[v]):
                res[i] = v
                i += 2
                if i >= n:
                    i = 1
        return res
```

Công thức nhớ nhanh: **đếm → giá trị nhiều nhất đứng đầu, mỗi giá trị là một đoạn liên tiếp → ghi vào chỗ chẵn rồi chỗ lẻ**.

### Checklist trước khi submit

- [ ] Giá trị phổ biến nhất đứng **đầu** thứ tự điền?
- [ ] Mỗi giá trị chiếm một **đoạn liên tiếp** (sắp xếp các giá trị khác nhau, không sắp từng phần tử)?
- [ ] Bắt đầu từ vị trí **chẵn** `0`, hết thì sang vị trí lẻ `1`?
- [ ] Chia đôi bằng `(n + 1) // 2` (nếu dùng lát cắt)?
- [ ] Test bằng **hàm kiểm tra hợp lệ** (nhiều đáp án đúng)?
- [ ] Đã thử: `n = 1`, mode đúng `ceil(n / 2)` với `n` lẻ và chẵn, nhiều giá trị cùng tần suất, `n = 10^4`?